 # UNC Attendance Model – NCAA Team Game Logs (UNC + Opponents) Pipeline

## Purpose
This notebook builds a modeling-ready dataset for predicting **attendance at games hosted at UNC’s stadium (Boshamer / Chapel Hill)**.

We start with a **GoHeels-scraped master dataset** (one row per UNC game with attendance) and augment each hosted game with:

- **UNC pre-game rolling performance features** from NCAA team game logs
- **Opponent pre-game rolling performance features** from NCAA team game logs (prior to the UNC matchup)

The NCAA stats are pulled via the `collegebaseball` library and cached locally to avoid repeated requests.

## Key outputs
- Cached raw NCAA game logs (Parquet), one file per `(team_id, season, variant)`
- A game-level feature table: **one row per UNC-hosted game**, with UNC + opponent features
- A final merged CSV that joins these features back onto the GoHeels dataset

## Notebook roadmap

1. Load GoHeels + mapping CSV  
2. Define standardizers  
3. Clean GoHeels hosted games + opponent mapping + DH ordering  
4. Apply Selenium headed patch  
5. Cache UNC + opponents gamelogs  
6. Build pre-game rolling features  
7. Crosswalk GoHeels hosted games → NCAA game_id  
8. Join UNC + opponent features and export final dataset  

In [ ]:
# this cell ensures that the updated version of the collegebaseball library is loaded.
from pathlib import Path
import sys
import pandas as pd

# Assumption: notebook is run from PROJECT_ROOT/SCRAPERS/performance/notebooks/
NB_DIR = Path.cwd().resolve()
PERFORMANCE_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
SCRAPERS_ROOT = PERFORMANCE_ROOT.parent
PROJECT_ROOT = SCRAPERS_ROOT.parent
VENDORED_CB_ROOT = PERFORMANCE_ROOT / "vendor" / "collegebaseball"   # this folder contains the "collegebaseball/" package

assert VENDORED_CB_ROOT.exists(), f"Vendored collegebaseball not found at: {VENDORED_CB_ROOT}"

# Put vendored package FIRST on sys.path (ahead of site-packages)
if str(VENDORED_CB_ROOT) not in sys.path:
    sys.path.insert(0, str(VENDORED_CB_ROOT))

# Now import and verify exactly where it came from
import collegebaseball
from collegebaseball import lookup, ncaa_scraper

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRAPERS_ROOT:", SCRAPERS_ROOT)
print("PERFORMANCE_ROOT:", PERFORMANCE_ROOT)
print("collegebaseball imported from:", Path(collegebaseball.__file__).resolve())

# Verify which seasons.csv you are using
seasons_path = Path(collegebaseball.__file__).resolve().parent / "data" / "seasons.csv"
print("seasons.csv:", seasons_path)
print("exists?", seasons_path.exists())


In [ ]:
import os
import pandas as pd
import sys
import numpy as np
import re
from urllib.parse import urlencode
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import random
from pathlib import Path


## 1) Inputs and working directories

This notebook assumes two local directories:

- **Scraper folder**: contains the GoHeels master CSV and `ncaa_school_mapping.csv`
- **collegebaseball folder**: contains the modified `collegebaseball` package plus `cache/`

We load:
- `goheels_df`: GoHeels-scraped master dataset (attendance target lives here)
- `ncaa_school_mapping`: mapping table to translate school names ↔ NCAA `school_id`

> Important: GoHeels does **not** contain an NCAA game_id. We will later build a crosswalk using date/opponent/DH ordering.

## 2) Using the modified `collegebaseball` package (local fork)

We use a locally-modified copy of the `collegebaseball` library to access `stats.ncaa.org`.

This fork exists because:
- newer seasons require updated season identifiers / headers
- some “advanced” metrics rely on weights not consistently available across all years
- `stats.ncaa.org` blocks many automated HTTP requests, requiring a headed Selenium workaround

This notebook prints the active Python executable and the path to the imported `collegebaseball` module so we can verify we are using the correct local version.

In [ ]:
# Assumption: notebook is run from PROJECT_ROOT/SCRAPERS/performance/notebooks/
NB_DIR = Path.cwd().resolve()                    # .../SCRAPERS/Game_Stat_Scraper/notebooks
GAME_SCRAPER_ROOT = NB_DIR.parent                # .../SCRAPERS/Game_Stat_Scraper
SCRAPERS_ROOT = GAME_SCRAPER_ROOT.parent         # .../SCRAPERS
PROJECT_ROOT = SCRAPERS_ROOT.parent              # .../992-S26-TeamA3

DATA_ROOT = GAME_SCRAPER_ROOT / "data"
EXPORTS_DIR = GAME_SCRAPER_ROOT / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

GOHEELS_CSV = DATA_ROOT / "unc_baseball_final.csv"
MAPPING_CSV = DATA_ROOT / "ncaa_school_mapping.csv"
FINAL_CSV = EXPORTS_DIR / "uncbaseball_season_performance_0331.csv"
UNC_FEATURES_PATH = EXPORTS_DIR / "unc_features.parquet"

VENDOR_CB_ROOT = GAME_SCRAPER_ROOT / "vendor" / "collegebaseball"
if not VENDOR_CB_ROOT.exists():
    raise FileNotFoundError(f"Expected vendored collegebaseball at: {VENDOR_CB_ROOT}")

if str(VENDOR_CB_ROOT) not in sys.path:
    sys.path.insert(0, str(VENDOR_CB_ROOT))

CACHE_UNC_DIR = VENDOR_CB_ROOT / "cache" / "unc"
CACHE_OPP_DIR = VENDOR_CB_ROOT / "cache" / "opponents"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRAPERS_ROOT:", SCRAPERS_ROOT)
print("PERFORMANCE_ROOT:", PERFORMANCE_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("GOHEELS_CSV exists?", GOHEELS_CSV.exists())
print("MAPPING_CSV exists?", MAPPING_CSV.exists())
print("CACHE_UNC_DIR exists?", CACHE_UNC_DIR.exists())
print("CACHE_OPP_DIR exists?", CACHE_OPP_DIR.exists())
print("FINAL_CSV:", FINAL_CSV)
print("UNC_FEATURES_PATH:", UNC_FEATURES_PATH)


In [ ]:
goheels_df = pd.read_csv(GOHEELS_CSV)

In [ ]:
ncaa_school_mapping = pd.read_csv(MAPPING_CSV)

## 3) Standardizing NCAA game-log schemas (2013–present continuity)

NCAA game logs vary slightly by season (some columns appear only in newer years).
To produce a stable multi-year modeling dataset, we standardize each variant:

### Batting
- Canonicalize double-play column naming:
  - older years: `DP`
  - newer years: `OPP DP`
- Store the unified version as `opp_dp`
- Drop `GDP` when present (inconsistent availability)

### Pitching
- Drop columns with inconsistent availability across early seasons:
  - `CG`, `pickoffs`

### Fielding
- Total Chances (`TC`) is standardized by recomputation everywhere:
  - `TC = PO + A + E`
  - overwrite existing `TC` or create it if missing

These transformations are applied **immediately after loading each gamelog** and also when exporting combined CSVs.

In [ ]:
# team game log variant standardizers
def standardize_batting(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Canonicalize DP naming
    if "OPP DP" in df.columns and "DP" not in df.columns:
        df = df.rename(columns={"OPP DP": "opp_dp"})
    elif "DP" in df.columns and "OPP DP" not in df.columns:
        df = df.rename(columns={"DP": "opp_dp"})
    elif "DP" in df.columns and "OPP DP" in df.columns:
        df = df.rename(columns={"OPP DP": "opp_dp"})
        df = df.drop(columns=["DP"])

    # 2) Drop GDP if present
    if "GDP" in df.columns:
        df = df.drop(columns=["GDP"])

    return df


def standardize_pitching(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_cols = [c for c in ["CG", "pickoffs"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df


def standardize_fielding(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for c in ["PO", "A", "E"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    po = df["PO"] if "PO" in df.columns else pd.Series([pd.NA] * len(df))
    a  = df["A"]  if "A"  in df.columns else pd.Series([pd.NA] * len(df))
    e  = df["E"]  if "E"  in df.columns else pd.Series([pd.NA] * len(df))

    all_nan = po.isna() & a.isna() & e.isna()
    tc = po.fillna(0) + a.fillna(0) + e.fillna(0)
    tc = tc.astype("float")
    tc[all_nan] = pd.NA

    df["TC"] = tc
    return df

## 4) GoHeels preprocessing: hosted-game filter, opponent mapping, and DH ordering

We restrict the GoHeels master dataset to games **hosted at UNC’s stadium**:
- venue contains “Boshamer” OR location city contains “Chapel Hill”
- remove rows with missing attendance
- remove known NCAA-mapping edge cases (e.g., opponents that cannot be pulled reliably)

### Opponent mapping (GoHeels name → NCAA `school_id`)
GoHeels opponent names are not guaranteed to match NCAA naming.
We normalize names and join against `ncaa_school_mapping.csv` using both:
- `ncaa_name`
- `bd_name`

We also apply a `MANUAL_OVERRIDES` dictionary for common aliases (e.g., “UNC Wilmington” → “UNCW”).

### Doubleheaders (GoHeels `dh_index`)
Doubleheaders exist and must be ordered deterministically for “prior games only” logic.
We compute a GoHeels `dh_index` within (season, game_date, opponent_norm), using start time when available.

In [ ]:
# go heels hosted cleaning + opponent mapping + goheels dh_index

def parse_goheels_date(date_str: str, season: int) -> pd.Timestamp:
    """
    GoHeels date examples: 'Feb 12(Fri)', 'May 27', etc.
    Remove '(Fri)' and parse with season year.
    """
    if pd.isna(date_str):
        return pd.NaT
    s = re.sub(r"\(.*?\)", "", str(date_str)).strip()
    dt = pd.to_datetime(f"{s} {int(season)}", format="%b %d %Y", errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors="coerce")
    return dt


def parse_goheels_start_time(x: str) -> pd.Timestamp:
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if s.lower() in {"canceled", "cancelled", "tba", "tbd", ""}:
        return pd.NaT
    return pd.to_datetime(s, format="%I:%M %p", errors="coerce")


def norm_name(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("&", "and")
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def attach_opponent_mapping(
    goheels_hosted_in: pd.DataFrame,
    ncaa_school_mapping: pd.DataFrame,
    manual_overrides: dict | None = None
) -> pd.DataFrame:
    df = goheels_hosted_in.copy()

    cols_to_drop = [
        "opponent_norm", "opponent_norm_for_map",
        "opp_school_id", "opp_ncaa_name", "opp_bd_name", "opp_mapped_flag",
        "ncaa_norm", "bd_norm",
        "ncaa_norm_map", "bd_norm_map",
        "ncaa_name", "bd_name", "school_id",
        "ncaa_name_from_ncaa", "bd_name_from_ncaa", "school_id_from_ncaa",
        "ncaa_name_from_bd", "bd_name_from_bd", "school_id_from_bd",
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

    df["opponent"] = df.get("opponent", np.nan)
    df["opponent_norm"] = df["opponent"].map(norm_name)

    mapping = ncaa_school_mapping[["ncaa_name", "bd_name", "school_id"]].copy()
    mapping["ncaa_norm"] = mapping["ncaa_name"].map(norm_name)
    mapping["bd_norm"]   = mapping["bd_name"].map(norm_name)

    if manual_overrides is None:
        manual_overrides = {}
    df["opponent_norm_for_map"] = df["opponent_norm"].map(lambda x: manual_overrides.get(x, x))

    map_ncaa = mapping.dropna(subset=["ncaa_norm"]).drop_duplicates("ncaa_norm")
    df = df.merge(
        map_ncaa[["ncaa_norm", "ncaa_name", "bd_name", "school_id"]].rename(
            columns={"ncaa_name": "ncaa_name_from_ncaa",
                     "bd_name": "bd_name_from_ncaa",
                     "school_id": "school_id_from_ncaa"}
        ),
        left_on="opponent_norm_for_map",
        right_on="ncaa_norm",
        how="left",
    )

    map_bd = mapping.dropna(subset=["bd_norm"]).drop_duplicates("bd_norm")
    df = df.merge(
        map_bd[["bd_norm", "ncaa_name", "bd_name", "school_id"]].rename(
            columns={"ncaa_name": "ncaa_name_from_bd",
                     "bd_name": "bd_name_from_bd",
                     "school_id": "school_id_from_bd"}
        ),
        left_on="opponent_norm_for_map",
        right_on="bd_norm",
        how="left",
    )

    df["opp_school_id"]   = df["school_id_from_ncaa"].combine_first(df["school_id_from_bd"])
    df["opp_ncaa_name"]   = df["ncaa_name_from_ncaa"].combine_first(df["ncaa_name_from_bd"])
    df["opp_bd_name"]     = df["bd_name_from_ncaa"].combine_first(df["bd_name_from_bd"])
    df["opp_mapped_flag"] = df["opp_school_id"].notna()

    return df


def clean_goheels_hosted(goheels_df: pd.DataFrame, ncaa_school_mapping: pd.DataFrame, manual_overrides: dict) -> pd.DataFrame:
    df = goheels_df.copy()

    # Standardize text cols used in filters
    df["venue"] = df.get("venue", "").fillna("").astype(str)

    # location_city may not exist in the updated scrape
    if "location_city" in df.columns:
        df["location_city"] = df["location_city"].fillna("").astype(str)
        is_chapel_hill = df["location_city"].str.contains("Chapel Hill", case=False, na=False)
    else:
        is_chapel_hill = False  # updated scrape removed this column

    # start_time may be missing in some scrapes; keep current behavior
    if "start_time" not in df.columns:
        df["start_time"] = np.nan

    # Hosted filter
    is_boshamer = df["venue"].str.contains("Boshamer", case=False, na=False)
    df = df.loc[is_boshamer | is_chapel_hill].copy()

    # Remove St. Francis (unloadable in NCAA pipeline)
    df = df.loc[df["opponent"] != "Saint Francis"].copy()

    # Keep historical rows only when attendance is present, but retain
    # current-season unplayed hosted games so future opponents still get cached.
    keep_hist = df["attendance"].notna()
    keep_future_2026 = pd.to_numeric(df["season"], errors="coerce").astype("Int64").eq(2026) & df["attendance"].isna()
    df = df.loc[keep_hist | keep_future_2026].copy()

    # Parse season + date -> game_date
    df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
    df["game_date_dt"] = [parse_goheels_date(d, s) for d, s in zip(df["date"], df["season"])]
    df["game_date"] = pd.to_datetime(df["game_date_dt"], errors="coerce")

    # Opponent mapping
    df = attach_opponent_mapping(df, ncaa_school_mapping, manual_overrides=manual_overrides)

    # DH index within GoHeels hosted
    df["start_time_dt"] = df["start_time"].map(parse_goheels_start_time)
    df["game_number"] = pd.to_numeric(df.get("game_number", np.nan), errors="coerce")

    df = df.sort_values(
        ["season", "game_date", "opponent_norm", "start_time_dt", "game_number"],
        na_position="last",
    )

    df["dh_index"] = (
        df.groupby(["season", "game_date", "opponent_norm"], dropna=False).cumcount() + 1
    )

    return df

## 5) Mapping NCAA opponent_name → `school_id` (for crosswalk support)

NCAA game logs include `opponent_name` but the `opponent_id` column is **not** directly callable with `ncaa_team_game_logs()`.
For consistent cache access, we map NCAA opponent names to the callable `school_id` using the same `ncaa_school_mapping.csv` and the same normalization logic as GoHeels.

We also maintain a small `NCAA_OVERRIDES` dictionary for NCAA-specific abbreviations (e.g., “App State” → “Appalachian State”).

In [ ]:
# mapping opponent names from ncaa logs to callable school_id
def build_name_to_school_id(ncaa_school_mapping: pd.DataFrame) -> dict[str, int]:
    m = ncaa_school_mapping[["school_id", "ncaa_name", "bd_name"]].copy()
    m["ncaa_norm"] = m["ncaa_name"].map(norm_name)
    m["bd_norm"]   = m["bd_name"].map(norm_name)

    # Prefer ncaa_name match, fallback to bd_name match
    out = {}
    for _, r in m.dropna(subset=["school_id"]).iterrows():
        sid = int(r["school_id"])
        if r["ncaa_norm"]:
            out.setdefault(r["ncaa_norm"], sid)
        if r["bd_norm"]:
            out.setdefault(r["bd_norm"], sid)
    return out

## 6) Headed Selenium patch (required for stats.ncaa.org)

`stats.ncaa.org` frequently blocks scripted HTTP requests (Access Denied / 403).
To make the pipeline reproducible, we monkeypatch the library’s request layer:

- Replace `requests.Session.get()` inside `collegebaseball.ncaa_scraper` with a Selenium-based getter
- Use a **headed** Chrome driver (NOT headless) because headless is more likely to be blocked
- Return the browser HTML (`page_source`) in a minimal Response-like object

This patch must be applied **before** running any NCAA scraping calls.

> After the pipeline completes, we restore the original `Session.get` and quit the Selenium driver.

In [ ]:
# applying patch to work around stats.ncaa.org website automation blockers

def _make_driver_headed():
    opts = webdriver.ChromeOptions()
    opts.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

_driver = _make_driver_headed()

_OrigSession = ncaa_scraper.Session
_orig_get = _OrigSession.get

class _FakeResp:
    def __init__(self, text, status_code=200, url=""):
        self.text = text
        self.status_code = status_code
        self.url = url

def _selenium_get(self, url, params=None, headers=None, **kwargs):
    full_url = url
    if params:
        full_url = f"{url}{'&' if '?' in url else '?'}{urlencode(params)}"

    _driver.get(full_url)
    # Wait until the page contains evidence of actual game rows.
    # game_id comes from contest links, so "/contests/" is a strong signal.
    for _ in range(30):  # ~15 seconds max
        html = _driver.page_source
        if ("/contests/" in html) and ("Schedule/Results" in html):
            break
        time.sleep(0.5)
    html = _driver.page_source

    if "Access Denied" in html or "You don't have permission" in html:
        print("Blocked URL:", full_url)
        return _FakeResp(html, status_code=403, url=full_url)

    return _FakeResp(html, status_code=200, url=full_url)

# apply patch
_OrigSession.get = _selenium_get
print("Patched collegebaseball Session.get -> Selenium headed browser.")

def unpatch_and_quit():
    _OrigSession.get = _orig_get
    try:
        _driver.quit()
    except Exception:
        pass
    print("Unpatched Session.get and quit Selenium driver.")

## 7) Caching NCAA logs (UNC + opponents) with retry/backoff

Scraping NCAA pages is slow and can fail transiently. We cache raw gamelogs to avoid repeated pulls.

### Cache layout
One file per `(school_id, season, variant)`:
- `cache/unc/school_<id>_season_<year>_<variant>.parquet`
- `cache/opponents/school_<id>_season_<year>_<variant>.parquet`

### Resilience
Each pull is wrapped in:
- retry loop with exponential backoff + jitter
- detection of blocked/empty pulls
- optional “skip vs raise” behavior on persistent failure

### Why this matters
This makes the pipeline resumable:
- reruns skip already-cached files (unless `force=True`)
- failures do not necessarily kill the entire run

In [ ]:
REQUIRED_META = [
    "date", "division", "extras", "field", "game_id", "innings_played",
    "opponent_id", "opponent_name", "result", "run_difference",
    "runs_allowed", "runs_scored", "school_id", "season_id"
]

def _looks_blocked_or_empty(df: pd.DataFrame) -> bool:
    """
    Treat 'bad pull' as retryable:
      - df is empty
      - required metadata columns missing
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return True
    if df.empty:
        return True
    missing = [c for c in REQUIRED_META if c not in df.columns]
    if missing:
        return True
    return False

def _is_retryable_exception(e: Exception) -> bool:
    """
    Common retryable errors when NCAA returns incomplete HTML or parsing fails.
    Keep this permissive; you're running a long job and want robustness.
    """
    msg = str(e).lower()
    retry_signals = [
        "no tables found",
        "empty data",
        "timed out",
        "timeout",
        "stale element",
        "disconnected",
        "connection aborted",
        "bad gateway",
        "502",
        "503",
        "access denied",
        "permission",
        "list index out of range",
        "index 0 is out of bounds",
        "keyerror",
    ]
    return any(s in msg for s in retry_signals)

In [ ]:
# ncaa team log pulling with caching (unc cached by season, opponents cached once per season
VARIANTS = ["batting", "pitching", "fielding"]

def _cache_path(cache_dir: str | Path, school_id: int, season: int, variant: str) -> Path:
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f"school_{school_id}_season_{season}_{variant}.parquet"


def _standardize_variant(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    if variant == "batting":
        df = standardize_batting(df)
    elif variant == "pitching":
        df = standardize_pitching(df)
    elif variant == "fielding":
        df = standardize_fielding(df)
    else:
        raise ValueError(f"Unknown variant: {variant}")
    return df


def _add_ncaa_order_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Uses guaranteed metadata: date, game_id.
    Adds:
      game_date (normalized Timestamp),
      dh_index within game_date (based on game_id ordering),
      sort columns.
    """
    out = df.copy()
    out["game_date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()

    out = out.sort_values(["game_date", "game_id"], na_position="last").reset_index(drop=True)
    out["dh_index"] = out.groupby(["game_date"], dropna=False).cumcount() + 1
    return out


def pull_team_variant_with_cache(
    *,
    ncaa_scraper,
    school_id: int,
    season: int,
    variant: str,
    cache_dir: str | Path,
    force: bool = False,
    max_tries: int = 4,
    base_sleep: float = 2.0,
    jitter: float = 0.5,
    on_fail: str = "skip",  # "skip" or "raise"
) -> pd.DataFrame:
    """
    Pull one (school_id, season, variant) gamelog with include_advanced=False,
    standardize it, add ordering cols, and cache it.

    Adds:
      - retry loop around ncaa_team_game_logs
      - detect empty/blocked pulls and retry
      - skip/raise behavior on persistent failure
    """
    p = _cache_path(cache_dir, school_id, season, variant)

    if p.exists() and not force:
        return pd.read_parquet(p)

    last_err = None

    for attempt in range(1, max_tries + 1):
        try:
            df = ncaa_scraper.ncaa_team_game_logs(
                school=school_id,
                season=season,
                variant=variant,
                include_advanced=False,
            )

            # Detect blocked/empty/incomplete pulls
            if df is None or not isinstance(df, pd.DataFrame) or df.empty:
                raise ValueError(
                    f"Bad pull (empty dataframe). school_id={school_id}, season={season}, variant={variant}"
                )
            missing = [c for c in REQUIRED_META if c not in df.columns]
            if missing:
                raise ValueError(
                    f"Bad pull (missing meta={missing}). school_id={school_id}, season={season}, variant={variant}"
                )

            # Standardize + order + cache
            df = _standardize_variant(df, variant)
            df = _add_ncaa_order_cols(df)

            df.to_parquet(p, index=False)
            print(f"[cached] school_id={school_id} season={season} variant={variant} rows={len(df)}")
            return df

        except Exception as e:
            last_err = e

            # If not retryable, break immediately
            if not _is_retryable_exception(e) and attempt == 1:
                # could still allow a couple tries, I'm being conservative here
                pass

            if attempt < max_tries:
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, jitter)
                print(
                    f"[retry {attempt}/{max_tries}] "
                    f"school_id={school_id} season={season} variant={variant} "
                    f"error={type(e).__name__}: {str(e)[:120]} ... sleeping {sleep_s:.1f}s"
                )
                time.sleep(sleep_s)
                continue
            else:
                # final failure
                msg = (
                    f"[FAIL] school_id={school_id} season={season} variant={variant} "
                    f"after {max_tries} tries. Last error: {type(last_err).__name__}: {last_err}"
                )
                if on_fail == "raise":
                    raise RuntimeError(msg) from last_err
                else:
                    print(msg)
                    # return empty df so caller can continue; do NOT cache empties by default
                    return pd.DataFrame()


def pull_team_season_bundle_with_cache(
    *,
    ncaa_scraper,
    school_id: int,
    season: int,
    cache_dir: str | Path,
    force: bool = False,
    on_fail: str = "skip",
) -> dict[str, pd.DataFrame]:
    bundle = {}
    for v in VARIANTS:
        dfv = pull_team_variant_with_cache(
            ncaa_scraper=ncaa_scraper,
            school_id=school_id,
            season=season,
            variant=v,
            cache_dir=cache_dir,
            force=force,
            on_fail=on_fail,
        )
        bundle[v] = dfv
    return bundle

In [ ]:
print("Python:", sys.executable)
print("collegebaseball:", collegebaseball.__file__)

## 8) Bulk caching plan: UNC by season, opponents once per season

We cache:
1) **UNC** gamelogs for every supported season (2013–present)
2) **Opponents** gamelogs only for teams that appear in UNC-hosted games, and only for the relevant seasons

We explicitly restrict to supported seasons (2013–2026 in this project) because:
- GoHeels includes older seasons
- the NCAA scraper is only validated in our pipeline for 2013+ right now

This step produces a complete cache that downstream feature engineering can use without re-scraping.

In [ ]:
# bulk caching plan
LATEST_SEASON_FORCE_REFRESH = True

def cache_unc_and_opponents(
    *,
    ncaa_scraper,
    goheels_hosted_clean: pd.DataFrame,
    ncaa_school_mapping: pd.DataFrame,
    cache_dir_unc: str | Path,
    cache_dir_opp: str | Path,
    unc_school_id: int,
    seasons: list[int] | None = None,
    force: bool = False,
    min_season: int = 2013,
    max_season: int = 2026,
    refresh_latest_season: bool = True,
):
    df = goheels_hosted_clean.copy()
    df = df.loc[df["opp_mapped_flag"]].copy()

    # restrict to supported NCAA seasons
    df["season"] = pd.to_numeric(df["season"], errors="coerce")
    df = df.loc[df["season"].between(min_season, max_season)].copy()

    if seasons is None:
        seasons = sorted(df["season"].dropna().astype(int).unique().tolist())
    else:
        # also enforce bounds if user passes seasons explicitly
        seasons = sorted({int(y) for y in seasons if min_season <= int(y) <= max_season})

    latest_season = max(seasons) if seasons else None

    # Identify the next upcoming hosted-game opponent in the latest season.
    # This is the only latest-season opponent whose current-season cache we
    # force-refresh on recurring runs. All other latest-season opponent caches
    # are left as-is unless missing.
    next_latest_opp_sid = None
    if refresh_latest_season and latest_season is not None:
        future_rows = df.loc[df["season"].eq(latest_season)].copy()

        if "attendance" in future_rows.columns:
            future_rows = future_rows.loc[future_rows["attendance"].isna()].copy()

        if "game_date" in future_rows.columns:
            future_rows["game_date"] = pd.to_datetime(future_rows["game_date"], errors="coerce")
        if "dh_index" not in future_rows.columns:
            future_rows["dh_index"] = 1
        future_rows["dh_index"] = pd.to_numeric(future_rows["dh_index"], errors="coerce").fillna(1)

        if "game_number" in future_rows.columns:
            future_rows["game_number"] = pd.to_numeric(future_rows["game_number"], errors="coerce")
        else:
            future_rows["game_number"] = np.nan

        future_rows = future_rows.loc[future_rows["opp_school_id"].notna()].copy()
        future_rows = future_rows.sort_values(
            ["game_date", "dh_index", "game_number"],
            na_position="last"
        )

        if len(future_rows):
            next_latest_opp_sid = int(future_rows["opp_school_id"].iloc[0])

    # 1) UNC cache by season
    for y in seasons:
        force_this_unc = force or (
            refresh_latest_season and latest_season is not None and int(y) == int(latest_season)
        )
        pull_team_season_bundle_with_cache(
            ncaa_scraper=ncaa_scraper,
            school_id=unc_school_id,
            season=int(y),
            cache_dir=cache_dir_unc,
            force=force_this_unc,
        )

    # 2) Opp cache once per opponent per season
    pairs = (
        df[["season", "opp_school_id"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["season", "opp_school_id"])
    )

    for season, opp_sid in pairs.itertuples(index=False):
        force_this_opp = force or (
            refresh_latest_season
            and latest_season is not None
            and int(season) == int(latest_season)
            and next_latest_opp_sid is not None
            and int(opp_sid) == int(next_latest_opp_sid)
        )
        pull_team_season_bundle_with_cache(
            ncaa_scraper=ncaa_scraper,
            school_id=int(opp_sid),
            season=int(season),
            cache_dir=cache_dir_opp,
            force=force_this_opp,
        )

        # also cache previous-season opponent logs (season-1) for preseason features
        prev_season = int(season) - 1
        if prev_season >= min_season:
            pull_team_season_bundle_with_cache(
                ncaa_scraper=ncaa_scraper,
                school_id=int(opp_sid),
                season=int(prev_season),
                cache_dir=cache_dir_opp,
                force=force,
            )


In [ ]:
# actually running everything above

# NCAA opponent_name quirks -> mapping CSV-friendly normalized names
NCAA_OVERRIDES = {
    norm_name("App State"): norm_name("Appalachian State")
}

NCAA_OVERRIDES.update({
    "queens nc": "queens"
})


# manual overrides
MANUAL_OVERRIDES = {
    norm_name("Virginia Commonwealth"): norm_name("VCU"),  # GoHeels full name -> mapping ncaa_name abbreviation
    norm_name("UNC Wilmington"): norm_name("UNCW"),        # common abbreviation in mapping ncaa_name
    norm_name("Cal State Fullerton"): norm_name("Cal St. Fullerton"),
    norm_name("Florida Atlantic"): norm_name("Fla. Atlantic"),
    norm_name("Southern Miss"): norm_name("Southern Miss."),
    norm_name("Middle Tennessee"): norm_name("Middle Tenn."),
    norm_name("South Florida"): norm_name("South Fla."),
    norm_name("St. John's"): norm_name("St. John's (NY)"),
    norm_name("Mount St. Mary's"): norm_name("Mount St. Mary's"),
    norm_name("Miami"): norm_name("Miami (FL)"),
    norm_name("Birmingham Southern"): norm_name("Birmingham-So."),
    norm_name("Queens"): norm_name("Queens (SAC)"),
    norm_name("Pace University"): norm_name("Pace"),
    norm_name("ACC Tournament"): "",  # not a school; leave unmapped
}


# 2) Clean GoHeels hosted dataset
goheels_hosted_clean = clean_goheels_hosted(
    goheels_df=goheels_df,
    ncaa_school_mapping=ncaa_school_mapping,
    manual_overrides=MANUAL_OVERRIDES
)




# 3) UNC school_id (from mapping CSV; pick the right row once)
unc_row = ncaa_school_mapping.loc[ncaa_school_mapping["ncaa_name"].map(norm_name).eq(norm_name("North Carolina"))].head(1)
unc_school_id = int(unc_row["school_id"].iloc[0])

# 4) Cache NCAA logs
cache_unc_and_opponents(
    ncaa_scraper=ncaa_scraper,
    goheels_hosted_clean=goheels_hosted_clean,
    ncaa_school_mapping=ncaa_school_mapping,
    cache_dir_unc=CACHE_UNC_DIR,
    cache_dir_opp=CACHE_OPP_DIR,
    unc_school_id=unc_school_id,
    seasons=None,   # auto from goheels_hosted_clean
    force=False,
    refresh_latest_season=LATEST_SEASON_FORCE_REFRESH,
)

unpatch_and_quit()

In [ ]:
# confirm Pitt's mapping row
pitt = ncaa_school_mapping.loc[ncaa_school_mapping["ncaa_name"].str.contains("Pittsburgh", na=False)].head(5)
display(pitt[["ncaa_name","bd_name","school_id"]])


In [ ]:
# verifying that we cached all files we expected
VARIANTS = ["batting", "pitching", "fielding"]

def expected_cache_paths(goheels_hosted_clean: pd.DataFrame,
                         unc_school_id: int,
                         cache_unc=CACHE_UNC_DIR,
                         cache_opp=CACHE_OPP_DIR,
                         min_season=2013,
                         max_season=2026):
    df = goheels_hosted_clean.copy()
    df = df.loc[df["opp_mapped_flag"]].copy()
    df["season"] = pd.to_numeric(df["season"], errors="coerce")
    df = df.loc[df["season"].between(min_season, max_season)].copy()

    seasons = sorted(df["season"].dropna().astype(int).unique().tolist())
    pairs = (
        df[["season", "opp_school_id"]]
        .dropna()
        .drop_duplicates()
        .astype({"season": int, "opp_school_id": int})
        .itertuples(index=False, name=None)
    )

    exp = []

    # UNC: every season x every variant
    for y in seasons:
        for v in VARIANTS:
            exp.append(Path(cache_unc) / f"school_{unc_school_id}_season_{y}_{v}.parquet")

    # Opponents: every (opp, season) x every variant
    for season, opp_sid in pairs:
        for v in VARIANTS:
            exp.append(Path(cache_opp) / f"school_{opp_sid}_season_{season}_{v}.parquet")

    return exp


exp = expected_cache_paths(goheels_hosted_clean, unc_school_id)

missing = [p for p in exp if not p.exists()]
print("Expected files:", len(exp))
print("Missing files:", len(missing))

# show a few missing (if any)
for p in missing[:25]:
    print("MISSING:", p)

In [ ]:
df_unc_bat_2019 = pd.read_parquet(CACHE_UNC_DIR / f"school_{unc_school_id}_season_2019_batting.parquet")
df_unc_bat_2019[["date","game_id","opponent_name","school_id","season_id","dh_index"]].head()

In [ ]:
opp_sid = int(goheels_hosted_clean.loc[goheels_hosted_clean["season"].eq(2019), "opp_school_id"].dropna().iloc[0])
df_opp_pit_2019 = pd.read_parquet(CACHE_OPP_DIR / f"school_{opp_sid}_season_2019_pitching.parquet")
df_opp_pit_2019[["date","game_id","opponent_name","school_id","season_id","dh_index"]].head()

### now shifting to the feature engineering with scraping complete

In [ ]:
# Assumption: notebook is run from PROJECT_ROOT/SCRAPERS/performance/notebooks/
NB_DIR = Path.cwd().resolve()
PERFORMANCE_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
SCRAPERS_ROOT = PERFORMANCE_ROOT.parent
PROJECT_ROOT = SCRAPERS_ROOT.parent

DATA_ROOT = PERFORMANCE_ROOT / "data"
NOTEBOOKS_DIR = PERFORMANCE_ROOT / "notebooks"
EXPORTS_DIR = PERFORMANCE_ROOT / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Vendored collegebaseball fork inside the repo
VENDOR_CB_ROOT = PERFORMANCE_ROOT / "vendor" / "collegebaseball"
if not VENDOR_CB_ROOT.exists():
    raise FileNotFoundError(f"Expected vendored collegebaseball at: {VENDOR_CB_ROOT}")

# Make vendored package importable
if str(VENDOR_CB_ROOT) not in sys.path:
    sys.path.insert(0, str(VENDOR_CB_ROOT))

GOHEELS_CSV = DATA_ROOT / "unc_baseball_final.csv"
MAPPING_CSV = DATA_ROOT / "ncaa_school_mapping.csv"
FINAL_CSV = EXPORTS_DIR / "uncbaseball_season_performance_0331.csv"
UNC_FEATURES_PATH = EXPORTS_DIR / "unc_features.parquet"

# Cache + exports (local artifacts; do not commit)
CACHE_UNC_DIR = VENDOR_CB_ROOT / "cache" / "unc"
CACHE_OPP_DIR = VENDOR_CB_ROOT / "cache" / "opponents"

# Backward-compatible aliases used later in the notebook
UNC_DIR = CACHE_UNC_DIR
OPP_DIR = CACHE_OPP_DIR

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRAPERS_ROOT:", SCRAPERS_ROOT)
print("PERFORMANCE_ROOT:", PERFORMANCE_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("GOHEELS_CSV exists?", GOHEELS_CSV.exists())
print("MAPPING_CSV exists?", MAPPING_CSV.exists())
print("CACHE_UNC_DIR exists?", CACHE_UNC_DIR.exists())
print("CACHE_OPP_DIR exists?", CACHE_OPP_DIR.exists())
print("FINAL_CSV:", FINAL_CSV)
print("UNC_FEATURES_PATH:", UNC_FEATURES_PATH)


In [ ]:
def load_team_season(team_dir):
    bat_files = sorted(team_dir.glob("*batting.parquet"))
    pit_files = sorted(team_dir.glob("*pitching.parquet"))
    fld_files = sorted(team_dir.glob("*fielding.parquet"))

    if not bat_files:
        raise ValueError(f"No batting files found in {team_dir}")
    if not pit_files:
        raise ValueError(f"No pitching files found in {team_dir}")
    if not fld_files:
        raise ValueError(f"No fielding files found in {team_dir}")

    bat = pd.concat([pd.read_parquet(f) for f in bat_files], ignore_index=True)
    pit = pd.concat([pd.read_parquet(f) for f in pit_files], ignore_index=True)
    fld = pd.concat([pd.read_parquet(f) for f in fld_files], ignore_index=True)

    bat = bat.drop_duplicates(subset="game_id").copy()
    pit = pit.drop_duplicates(subset="game_id").copy()
    fld = fld.drop_duplicates(subset="game_id").copy()

    df = bat.merge(pit, on="game_id", how="left", indicator="_pit_merge")
    df = df.merge(fld, on="game_id", how="left", indicator="_fld_merge")

    return df

In [ ]:
def prepare_team_df(df):
    drop_cols = [
        "date_pit", "date_fld",
        "field_pit", "field_fld",
        "division_pit", "division_fld"
    ]
    drop_cols = [c for c in drop_cols if c in df.columns]

    df = df.drop(columns=drop_cols).copy()

    if "game_id" in df.columns and df["game_id"].duplicated().any():
        raise ValueError("Duplicate game_id values remain in prepare_team_df")

    sort_cols = [c for c in ["season", "game_date", "dh_index", "game_id"] if c in df.columns]
    df = df.sort_values(sort_cols).reset_index(drop=True)

    df["game_number"] = df.groupby("season").cumcount() + 1
    df["games_played"] = df["game_number"] - 1

    df["win"] = df["result"].fillna("").str.lower().eq("win").astype(int)

    return df

In [ ]:
CORE_STATS = {
    "runs_pg": "runs_scored",
    "runs_allowed_pg": "runs_allowed",
    "hr_pg": "HR",
    "errors_pg": "E",
    "k_pg": "K",
    "so_pg": "SO",
    "hr_allowed_pg": "HR-A",
    "run_diff_pg": "run_difference",
    "idp_pg": "IDP",
}

In [ ]:
def add_pregame_averages(df):
    df = df.copy()

    for new_col, source in CORE_STATS.items():
        if source not in df.columns:
            continue

        df[new_col] = (
            df.groupby("season")[source]
            .transform(lambda x: x.shift(1).expanding().mean())
        )

    df["win_pct"] = (
        df.groupby("season")["win"]
        .transform(lambda x: x.shift(1).expanding().mean())
    )

    return df


In [ ]:
def add_derived_features(df):
    df = df.copy()

    if {"hr_pg", "runs_pg"}.issubset(df.columns):
        df["power_index_avg"] = df["hr_pg"] + 0.2 * df["runs_pg"]

    if {"run_diff_pg", "win_pct"}.issubset(df.columns):
        df["team_strength"] = df["run_diff_pg"] + 2 * df["win_pct"]

    if {"SB", "CS"}.issubset(df.columns):
        sb_cum = df.groupby("season")["SB"].transform(lambda s: s.shift(1).cumsum())
        cs_cum = df.groupby("season")["CS"].transform(lambda s: s.shift(1).cumsum())
        attempts = sb_cum + cs_cum

        df["sb_success"] = np.where(
            df["games_played"].eq(0),
            np.nan,                         # season opener: no prior games
            np.where(
                attempts > 0,
                sb_cum / attempts,          # normal pregame success rate
                0.0                         # prior games exist, but no prior steal attempts yet
            )
        )

    return df

In [ ]:
def add_rolling_features(df):
    df = df.copy()

    df["wins_last3"] = (
        df.groupby("season")["win"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).sum())
    )

    df["wins_last5"] = (
        df.groupby("season")["win"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )

    for stat in ["runs_scored", "HR", "team_strength"]:
        if stat not in df.columns:
            continue

        name = stat.replace("_scored", "")

        df[f"{name}_last3"] = (
            df.groupby("season")[stat]
            .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        )

        df[f"{name}_last5"] = (
            df.groupby("season")[stat]
            .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        )

    return df

In [ ]:
def compute_previous_season_stats(df):
    season_summary = (
        df.groupby("season", as_index=False)
        .agg({
            "runs_scored": "mean",
            "runs_allowed": "mean",
            "HR": "mean",
            "E": "mean",
            "K": "mean",
            "SO": "mean",
            "HR-A": "mean",
            "IDP": "mean",
            "run_difference": "mean",
            "win": "mean",
        })
        .rename(columns={
            "runs_scored": "runs_pg_prev_season",
            "runs_allowed": "runs_allowed_pg_prev_season",
            "HR": "hr_pg_prev_season",
            "E": "errors_pg_prev_season",
            "K": "k_pg_prev_season",
            "SO": "so_pg_prev_season",
            "HR-A": "hr_allowed_pg_prev_season",
            "IDP": "idp_pg_prev_season",
            "run_difference": "run_diff_pg_prev_season",
            "win": "win_pct_prev_season",
        })
    )

    season_summary["power_index_avg_prev_season"] = (
        season_summary["hr_pg_prev_season"]
        + 0.2 * season_summary["runs_pg_prev_season"]
    )

    season_summary["team_strength_prev_season"] = (
        season_summary["run_diff_pg_prev_season"]
        + 2 * season_summary["win_pct_prev_season"]
    )

    season_summary["season"] = season_summary["season"] + 1

    df = df.merge(season_summary, on="season", how="left")

    return df

In [ ]:
def build_team_features(team_dir):
    df = load_team_season(team_dir)
    df = prepare_team_df(df)

    df = add_pregame_averages(df)
    df = compute_previous_season_stats(df)
    # df = apply_early_fallback(df)

    df = add_derived_features(df)
    df = add_rolling_features(df)

    return df

In [ ]:
unc = build_team_features(UNC_DIR)

unc.to_parquet(UNC_FEATURES_PATH)
print(f"saved UNC feature parquet -> {UNC_FEATURES_PATH}")


In [ ]:

mapping = pd.read_csv(MAPPING_CSV)

mapping["ncaa_name"] = mapping["ncaa_name"].astype(str).str.strip()
mapping["bd_name"] = mapping["bd_name"].astype(str).str.strip()

def norm_name(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("&", "and")
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def build_name_to_school_id(ncaa_school_mapping: pd.DataFrame) -> dict[str, int]:
    m = ncaa_school_mapping[["school_id", "ncaa_name", "bd_name"]].copy()
    m["ncaa_norm"] = m["ncaa_name"].map(norm_name)
    m["bd_norm"] = m["bd_name"].map(norm_name)

    out = {}
    for _, r in m.dropna(subset=["school_id"]).iterrows():
        sid = int(r["school_id"])
        if r["ncaa_norm"]:
            out.setdefault(r["ncaa_norm"], sid)
        if r["bd_norm"]:
            out.setdefault(r["bd_norm"], sid)
    return out

def map_opponent_names_to_school_id(
    names: pd.Series,
    ncaa_school_mapping: pd.DataFrame,
    manual_overrides: dict,
    ncaa_overrides: dict | None = None,
) -> pd.Series:
    if ncaa_overrides is None:
        ncaa_overrides = {}

    name_to_id = build_name_to_school_id(ncaa_school_mapping)

    normed = names.map(norm_name)
    normed = normed.map(lambda x: ncaa_overrides.get(x, x))
    normed = normed.map(lambda x: manual_overrides.get(x, x))

    return normed.map(lambda x: name_to_id.get(x, np.nan))

def parse_goheels_date(date_str: str, season: int) -> pd.Timestamp:
    if pd.isna(date_str):
        return pd.NaT
    s = re.sub(r"\(.*?\)", "", str(date_str)).strip()
    dt = pd.to_datetime(f"{s} {int(season)}", format="%b %d %Y", errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors="coerce")
    return dt

def parse_goheels_start_time(x: str) -> pd.Timestamp:
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if s.lower() in {"canceled", "cancelled", "tba", "tbd", ""}:
        return pd.NaT
    return pd.to_datetime(s, format="%I:%M %p", errors="coerce")

def attach_opponent_mapping(
    goheels_hosted_in: pd.DataFrame,
    ncaa_school_mapping: pd.DataFrame,
    manual_overrides: dict | None = None
) -> pd.DataFrame:
    df = goheels_hosted_in.copy()

    cols_to_drop = [
        "opponent_norm", "opponent_norm_for_map",
        "opp_school_id", "opp_ncaa_name", "opp_bd_name", "opp_mapped_flag",
        "ncaa_norm", "bd_norm",
        "ncaa_norm_map", "bd_norm_map",
        "ncaa_name", "bd_name", "school_id",
        "ncaa_name_from_ncaa", "bd_name_from_ncaa", "school_id_from_ncaa",
        "ncaa_name_from_bd", "bd_name_from_bd", "school_id_from_bd",
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

    df["opponent"] = df.get("opponent", np.nan)
    df["opponent_norm"] = df["opponent"].map(norm_name)

    mapping_local = ncaa_school_mapping[["ncaa_name", "bd_name", "school_id"]].copy()
    mapping_local["ncaa_norm"] = mapping_local["ncaa_name"].map(norm_name)
    mapping_local["bd_norm"] = mapping_local["bd_name"].map(norm_name)

    if manual_overrides is None:
        manual_overrides = {}
    df["opponent_norm_for_map"] = df["opponent_norm"].map(lambda x: manual_overrides.get(x, x))

    map_ncaa = mapping_local.dropna(subset=["ncaa_norm"]).drop_duplicates("ncaa_norm")
    df = df.merge(
        map_ncaa[["ncaa_norm", "ncaa_name", "bd_name", "school_id"]].rename(
            columns={"ncaa_name": "ncaa_name_from_ncaa",
                     "bd_name": "bd_name_from_ncaa",
                     "school_id": "school_id_from_ncaa"}
        ),
        left_on="opponent_norm_for_map",
        right_on="ncaa_norm",
        how="left",
    )

    map_bd = mapping_local.dropna(subset=["bd_norm"]).drop_duplicates("bd_norm")
    df = df.merge(
        map_bd[["bd_norm", "ncaa_name", "bd_name", "school_id"]].rename(
            columns={"ncaa_name": "ncaa_name_from_bd",
                     "bd_name": "bd_name_from_bd",
                     "school_id": "school_id_from_bd"}
        ),
        left_on="opponent_norm_for_map",
        right_on="bd_norm",
        how="left",
    )

    df["opp_school_id"] = df["school_id_from_ncaa"].combine_first(df["school_id_from_bd"])
    df["opp_ncaa_name"] = df["ncaa_name_from_ncaa"].combine_first(df["ncaa_name_from_bd"])
    df["opp_bd_name"] = df["bd_name_from_ncaa"].combine_first(df["bd_name_from_bd"])
    df["opp_mapped_flag"] = df["opp_school_id"].notna()

    return df

def clean_goheels_hosted(
    goheels_df: pd.DataFrame,
    ncaa_school_mapping: pd.DataFrame,
    manual_overrides: dict,
) -> pd.DataFrame:
    df = goheels_df.copy()

    df["venue"] = df.get("venue", "").fillna("").astype(str)

    if "location_city" in df.columns:
        df["location_city"] = df["location_city"].fillna("").astype(str)
        is_chapel_hill = df["location_city"].str.contains("Chapel Hill", case=False, na=False)
    else:
        is_chapel_hill = False

    if "start_time" not in df.columns:
        df["start_time"] = np.nan

    is_boshamer = df["venue"].str.contains("Boshamer", case=False, na=False)
    df = df.loc[is_boshamer | is_chapel_hill].copy()

    df = df.loc[df["opponent"] != "Saint Francis"].copy()

    keep_hist = df["attendance"].notna()
    keep_future_2026 = pd.to_numeric(df["season"], errors="coerce").astype("Int64").eq(2026) & df["attendance"].isna()
    df = df.loc[keep_hist | keep_future_2026].copy()

    df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
    df["game_date_dt"] = [parse_goheels_date(d, s) for d, s in zip(df["date"], df["season"])]
    df["game_date"] = pd.to_datetime(df["game_date_dt"], errors="coerce")

    df = attach_opponent_mapping(df, ncaa_school_mapping, manual_overrides=manual_overrides)

    df["start_time_dt"] = df["start_time"].map(parse_goheels_start_time)
    df["game_number"] = pd.to_numeric(df.get("game_number", np.nan), errors="coerce")

    df = df.sort_values(
        ["season", "game_date", "opponent_norm", "start_time_dt", "game_number"],
        na_position="last",
    )

    df["dh_index"] = (
        df.groupby(["season", "game_date", "opponent_norm"], dropna=False).cumcount() + 1
    )

    return df

NCAA_OVERRIDES = {
    norm_name("App State"): norm_name("Appalachian State"),
    "queens nc": "queens",
}

GOHEELS_MANUAL_OVERRIDES = {
    norm_name("Virginia Commonwealth"): norm_name("VCU"),
    norm_name("UNC Wilmington"): norm_name("UNCW"),
    norm_name("Cal State Fullerton"): norm_name("Cal St. Fullerton"),
    norm_name("Florida Atlantic"): norm_name("Fla. Atlantic"),
    norm_name("Southern Miss"): norm_name("Southern Miss."),
    norm_name("Middle Tennessee"): norm_name("Middle Tenn."),
    norm_name("South Florida"): norm_name("South Fla."),
    norm_name("St. John's"): norm_name("St. John's (NY)"),
    norm_name("Mount St. Mary's"): norm_name("Mount St. Mary's"),
    norm_name("Miami"): norm_name("Miami (FL)"),
    norm_name("Birmingham Southern"): norm_name("Birmingham-So."),
    norm_name("Queens"): norm_name("Queens (SAC)"),
    norm_name("Pace University"): norm_name("Pace"),
    norm_name("ACC Tournament"): "",
}

goheels_df = pd.read_csv(GOHEELS_CSV)
goheels_hosted_clean = clean_goheels_hosted(
    goheels_df=goheels_df,
    ncaa_school_mapping=mapping,
    manual_overrides=GOHEELS_MANUAL_OVERRIDES,
)

unc["opp_school_id"] = map_opponent_names_to_school_id(
    unc["opponent_name"],
    mapping,
    GOHEELS_MANUAL_OVERRIDES,
    ncaa_overrides=NCAA_OVERRIDES,
)


In [ ]:

def build_goheels_spine(goheels_hosted_clean: pd.DataFrame) -> pd.DataFrame:
    gh = goheels_hosted_clean.copy()
    gh = gh.loc[gh["season"].between(2013, 2026)].copy()
    gh["game_date"] = pd.to_datetime(gh["game_date"], errors="coerce").dt.normalize()
    gh["season"] = gh["season"].astype(int)
    gh["dh_index"] = gh["dh_index"].astype(int)

    keep = [
        "season", "game_date", "opponent", "dh_index",
        "opp_school_id", "opp_mapped_flag", "unc_score", "opponent_score"
    ]
    return gh[keep].copy()

def build_unc_ncaa_spine(unc_df: pd.DataFrame) -> pd.DataFrame:
    spine = unc_df.copy()
    spine["game_date"] = pd.to_datetime(spine["game_date"], errors="coerce").dt.normalize()
    spine["season"] = spine["season"].astype(int)
    spine["dh_index"] = spine["dh_index"].astype(int)

    keep = ["season", "game_date", "opponent_name", "dh_index", "game_id", "opp_school_id"]
    return spine[keep].copy()

def fill_missing_game_id_by_nearest_date(
    xw: pd.DataFrame,
    ncaa_spine: pd.DataFrame,
    *,
    days_tol: int = 2,
    strict: bool = False,
) -> pd.DataFrame:
    out = xw.copy()

    used = set(out.loc[out["game_id"].notna(), "game_id"].astype(int).tolist())

    ns = ncaa_spine.copy()
    ns = ns.dropna(subset=["opp_school_id", "game_date", "game_id"]).copy()
    ns["opp_school_id"] = ns["opp_school_id"].astype(int)
    ns["game_id"] = ns["game_id"].astype(int)

    for i, r in out.loc[out["game_id"].isna()].iterrows():
        if pd.isna(r["opp_school_id"]) or pd.isna(r["game_date"]):
            continue

        opp = int(r["opp_school_id"])
        gd = pd.to_datetime(r["game_date"]).normalize()

        cand = ns.loc[ns["opp_school_id"].eq(opp)].copy()
        cand["day_diff"] = (cand["game_date"] - gd).abs().dt.days
        cand = cand.loc[cand["day_diff"].le(days_tol)]
        cand = cand.loc[~cand["game_id"].isin(used)]
        cand = cand.sort_values(["day_diff", "game_date", "dh_index", "game_id"])

        if len(cand):
            gid = int(cand.iloc[0]["game_id"])
            out.at[i, "game_id"] = gid
            if "opponent_name" in out.columns and pd.isna(out.at[i, "opponent_name"]):
                out.at[i, "opponent_name"] = cand.iloc[0].get("opponent_name", out.at[i, "opponent_name"])
            used.add(gid)

    bad = out[out["game_id"].isna()]
    if len(bad) and strict:
        raise ValueError(
            f"Missing game_id after ±{days_tol}d fallback: {len(bad)} rows.\n"
            f"{bad[['season','game_date','opponent','dh_index','opp_school_id']].head(20)}"
        )

    return out

unc_full = unc.copy()

goheels_spine = build_goheels_spine(goheels_hosted_clean)
unc_ncaa_spine = build_unc_ncaa_spine(unc_full)

unc_crosswalk = goheels_spine.merge(
    unc_ncaa_spine,
    on=["season", "game_date", "opp_school_id", "dh_index"],
    how="left",
)

unc_crosswalk = fill_missing_game_id_by_nearest_date(
    unc_crosswalk,
    unc_ncaa_spine,
    days_tol=2,
    strict=False,
)

unc_feature_lookup = unc_full.drop(
    columns=["season", "game_date", "dh_index", "opp_school_id", "opponent_name"],
    errors="ignore",
).drop_duplicates(subset=["game_id"]).copy()

unc = (
    unc_crosswalk.merge(
        unc_feature_lookup,
        on="game_id",
        how="left",
    )
    .sort_values(["season", "game_date", "dh_index"])
    .reset_index(drop=True)
)

print("GoHeels hosted rows:", len(goheels_spine))
print("Hosted rows with NCAA game_id:", int(unc["game_id"].notna().sum()))
print("Rows in UNC feature base after hosted-game alignment:", len(unc))


In [ ]:
def compute_previous_season_stats_by_team(df):
    season_summary = (
        df.groupby(["school_id", "season"], as_index=False)
        .agg({
            "runs_scored": "mean",
            "runs_allowed": "mean",
            "HR": "mean",
            "E": "mean",
            "K": "mean",
            "SO": "mean",
            "HR-A": "mean",
            "IDP": "mean",
            "run_difference": "mean",
            "win": "mean",
        })
        .rename(columns={
            "runs_scored": "runs_pg_prev_season",
            "runs_allowed": "runs_allowed_pg_prev_season",
            "HR": "hr_pg_prev_season",
            "E": "errors_pg_prev_season",
            "K": "k_pg_prev_season",
            "SO": "so_pg_prev_season",
            "HR-A": "hr_allowed_pg_prev_season",
            "IDP": "idp_pg_prev_season",
            "run_difference": "run_diff_pg_prev_season",
            "win": "win_pct_prev_season",
        })
    )

    season_summary["power_index_avg_prev_season"] = (
        season_summary["hr_pg_prev_season"]
        + 0.2 * season_summary["runs_pg_prev_season"]
    )

    season_summary["team_strength_prev_season"] = (
        season_summary["run_diff_pg_prev_season"]
        + 2 * season_summary["win_pct_prev_season"]
    )

    season_summary["season"] = season_summary["season"] + 1

    df = df.merge(
        season_summary,
        on=["school_id", "season"],
        how="left"
    )

    return df

In [ ]:
def build_all_opponent_features():
    bat_files = sorted(OPP_DIR.glob("*_batting.parquet"))
    opp_dfs = []

    for bat_file in bat_files:
        name = bat_file.name.split("_")
        school_id = int(name[1])

        pit_file = bat_file.with_name(bat_file.name.replace("batting", "pitching"))
        fld_file = bat_file.with_name(bat_file.name.replace("batting", "fielding"))

        try:
            bat = pd.read_parquet(bat_file)
            pit = pd.read_parquet(pit_file)
            fld = pd.read_parquet(fld_file)

            bat = bat.drop_duplicates(subset="game_id").copy()
            pit = pit.drop_duplicates(subset="game_id").copy()
            fld = fld.drop_duplicates(subset="game_id").copy()

            df = bat.merge(pit, on="game_id", how="left")
            df = df.merge(fld, on="game_id", how="left")

            df["school_id"] = school_id

            df = prepare_team_df(df)
            df = add_pregame_averages(df)
            df = add_derived_features(df)
            df = add_rolling_features(df)

            opp_dfs.append(df)

        except Exception as e:
            print("skip:", bat_file.name, e)

    if not opp_dfs:
        raise RuntimeError("No opponent datasets created")

    opp = pd.concat(opp_dfs, ignore_index=True)
    opp = opp.sort_values(["school_id", "season", "game_date"]).reset_index(drop=True)

    opp = compute_previous_season_stats_by_team(opp)

    return opp


In [ ]:
opp = build_all_opponent_features()

In [ ]:
def keep_only_plain_opponent_name(df):
    cols = [c for c in df.columns if "opponent_name" in c]
    drop_cols = [c for c in cols if c != "opponent_name"]
    return df.drop(columns=drop_cols, errors="ignore").copy()

# clean the source frames first
unc = keep_only_plain_opponent_name(unc)
opp = keep_only_plain_opponent_name(opp)

print("UNC opponent name cols:", [c for c in unc.columns if "opponent_name" in c])
print("OPP opponent name cols:", [c for c in opp.columns if "opponent_name" in c])

In [ ]:
# --- exact NCAA game_id merge to mirror final_game_stat_scraper ---
opp_vs_unc = opp.loc[opp["opponent_name"] == "North Carolina"].copy()

# Drop leftover duplicate/source-merge columns that can cause name collisions
drop_cols = [
    c for c in opp_vs_unc.columns
    if c.endswith("_x") or c.endswith("_y")
]

# Keep only the UNC-side hosted-game join key after the merge.
# On the opponent side, school_id is the opponent's own school id and would be renamed
# to opp_school_id, which collides with UNC's existing hosted-game join key.
drop_cols += [c for c in ["opp_school_id", "school_id"] if c in opp_vs_unc.columns]

opp_vs_unc = opp_vs_unc.drop(columns=drop_cols, errors="ignore").copy()

exclude = ["game_id", "opponent_name"]

rename_cols = {
    c: f"opp_{c}"
    for c in opp_vs_unc.columns
    if c not in exclude
}

opp_vs_unc = opp_vs_unc.rename(columns=rename_cols)
opp_vs_unc = opp_vs_unc.drop(columns=["opponent_name"], errors="ignore")

matchups = unc.merge(
    opp_vs_unc,
    on="game_id",
    how="left"
)

strict_missing_count = matchups["opp_team_strength"].isna().sum()

print("After exact game_id merge:")
print("Hosted UNC rows:", len(matchups))
print("Missing opponent matches:", strict_missing_count)


In [ ]:

# Opponent features are joined by exact NCAA game_id to match final_game_stat_scraper.
# No season/date fallback is applied here because that can incorrectly borrow same-day
# doubleheader values across games.

print("Still missing after exact game_id merge:", matchups["opp_team_strength"].isna().sum())


In [ ]:
matchups["strength_diff"] = (
    matchups["team_strength"]
    - matchups["opp_team_strength"]
)

matchups["win_pct_diff"] = (
    matchups["win_pct"]
    - matchups["opp_win_pct"]
)

# matchups["run_diff_pg_diff"] = (
#     matchups["run_diff_pg"]
#     - matchups["opp_run_diff_pg"]
# )

# matchups["runs_pg_diff"] = (
#     matchups["runs_pg"]
#     - matchups["opp_runs_pg"]
# )

# matchups["hr_pg_diff"] = (
#     matchups["hr_pg"]
#     - matchups["opp_hr_pg"]
# )

In [ ]:
pd.set_option("display.max_info_columns", 300)

merge_keys = ["season", "game_date", "dh_index", "opp_school_id"]

unc_pregame_core = [
    "runs_pg",
    "runs_allowed_pg",
    "hr_pg",
    "errors_pg",
    "k_pg",
    "so_pg",
    "hr_allowed_pg",
    "idp_pg",
    "run_diff_pg",
    "win_pct",
]

unc_derived = [
    "so_bf_rate_avg",
    "error_rate_avg",
    "power_index_avg",
    "team_strength",
    "sb_success",
]

unc_recent = [
    "wins_last3",
    "wins_last5",
    "runs_last3",
    "runs_last5",
    "HR_last3",
    "HR_last5",
    "team_strength_last3",
    "team_strength_last5",
]

unc_prev_season = [
    "runs_pg_prev_season",
    "runs_allowed_pg_prev_season",
    "hr_pg_prev_season",
    "errors_pg_prev_season",
    "k_pg_prev_season",
    "so_pg_prev_season",
    "hr_allowed_pg_prev_season",
    "idp_pg_prev_season",
    "run_diff_pg_prev_season",
    "win_pct_prev_season",
    "power_index_prev_season",
    "team_strength_prev_season",
]

opp_pregame_core = [
    "opp_runs_pg",
    "opp_runs_allowed_pg",
    "opp_hr_pg",
    "opp_errors_pg",
    "opp_k_pg",
    "opp_so_pg",
    "opp_hr_allowed_pg",
    "opp_idp_pg",
    "opp_run_diff_pg",
    "opp_win_pct",
]

opp_derived = [
    "opp_so_bf_rate_avg",
    "opp_error_rate_avg",
    "opp_power_index_avg",
    "opp_team_strength",
    "opp_sb_success",
]

opp_recent = [
    "opp_wins_last3",
    "opp_wins_last5",
    "opp_runs_last3",
    "opp_runs_last5",
    "opp_HR_last3",
    "opp_HR_last5",
    "opp_team_strength_last3",
    "opp_team_strength_last5",
]

opp_prev_season = [
    "opp_runs_pg_prev_season",
    "opp_runs_allowed_pg_prev_season",
    "opp_hr_pg_prev_season",
    "opp_errors_pg_prev_season",
    "opp_k_pg_prev_season",
    "opp_so_pg_prev_season",
    "opp_hr_allowed_pg_prev_season",
    "opp_idp_pg_prev_season",
    "opp_run_diff_pg_prev_season",
    "opp_win_pct_prev_season",
    "opp_power_index_prev_season",
    "opp_team_strength_prev_season",
]

matchup_diffs = [
    "strength_diff",
    "win_pct_diff",
    # "run_diff_pg_diff",
    # "runs_pg_diff",
    # "hr_pg_diff",
]

coverage_flags = [
    "has_opponent_features",
]

ordered_feature_cols = (
    ["game_id"]
    + unc_pregame_core
    + unc_prev_season
    + unc_derived
    + unc_recent
    + opp_pregame_core
    + opp_prev_season
    + opp_derived
    + opp_recent
    + matchup_diffs
    + coverage_flags
)

ordered_feature_cols = [c for c in ordered_feature_cols if c in matchups.columns]

if "has_opponent_features" not in matchups.columns:
    matchups["has_opponent_features"] = (~matchups["opp_team_strength"].isna()).astype(int)
    if "has_opponent_features" not in ordered_feature_cols:
        ordered_feature_cols.append("has_opponent_features")


def _first_non_null(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan


def build_unc_preseason_lookup(unc_full: pd.DataFrame, prev_cols: list[str]) -> pd.DataFrame:
    cols = [c for c in ["season"] + prev_cols if c in unc_full.columns]
    out = (
        unc_full[cols]
        .groupby("season", as_index=False)
        .agg({c: _first_non_null for c in cols if c != "season"})
    )
    return out


def build_opp_preseason_lookup(opp: pd.DataFrame, prev_cols: list[str]) -> pd.DataFrame:
    cols = [c for c in ["school_id", "season"] + prev_cols if c in opp.columns]
    out = (
        opp[cols]
        .groupby(["school_id", "season"], as_index=False)
        .agg({c: _first_non_null for c in cols if c not in {"school_id", "season"}})
        .rename(columns={"school_id": "opp_school_id"})
    )
    return out



def _make_synthetic_game_id(work: pd.DataFrame, *, season: int, game_date, dh_index: int, school_id: int | None = None):
    if "game_id" not in work.columns:
        return None

    gid = work["game_id"].dropna()
    if gid.empty:
        return f"__future_{season}_{pd.to_datetime(game_date, errors='coerce').date()}_{dh_index}_{school_id if school_id is not None else 'unc'}"

    try:
        gid_num = pd.to_numeric(gid, errors="raise")
        next_id = int(gid_num.max()) + 1

        if gid.dtype == object and gid.map(lambda x: isinstance(x, str)).all():
            return str(next_id)
        return next_id
    except Exception:
        return f"__future_{season}_{pd.to_datetime(game_date, errors='coerce').date()}_{dh_index}_{school_id if school_id is not None else 'unc'}"


def add_derived_features_next_snapshot(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if {"hr_pg", "runs_pg"}.issubset(df.columns):
        df["power_index_avg"] = df["hr_pg"] + 0.2 * df["runs_pg"]

    if {"run_diff_pg", "win_pct"}.issubset(df.columns):
        df["team_strength"] = df["run_diff_pg"] + 2 * df["win_pct"]

    if {"SB", "CS"}.issubset(df.columns):
        sb_cum = df.groupby("season")["SB"].transform(lambda s: s.shift(1).cumsum())
        cs_cum = df.groupby("season")["CS"].transform(lambda s: s.shift(1).cumsum())
        attempts = sb_cum + cs_cum

        safe_rate = pd.to_numeric(sb_cum, errors="coerce").div(
            pd.to_numeric(attempts, errors="coerce").replace(0, np.nan)
        )

        if "games_played" in df.columns:
            df["sb_success"] = np.where(
                df["games_played"].eq(0),
                np.nan,
                safe_rate.fillna(0.0),
            )
        else:
            df["sb_success"] = safe_rate.fillna(0.0)

    return df


def build_next_team_snapshot(
    team_df: pd.DataFrame,
    *,
    season: int,
    game_date,
    dh_index: int,
    prev_stats_func,
    school_id: int | None = None,
) -> pd.DataFrame:
    work = team_df.copy()

    if school_id is not None:
        work = work.loc[work["school_id"].eq(int(school_id))].copy()

    # IMPORTANT:
    # For the next-game snapshot, only use the target season.
    # Preseason / prior-season features are already handled separately.
    if "season" in work.columns:
        work = work.loc[work["season"].eq(int(season))].copy()

    game_date_ts = pd.to_datetime(game_date, errors="coerce")
    if pd.isna(game_date_ts):
        return pd.DataFrame(columns=team_df.columns)

    game_date_norm = pd.Timestamp(game_date_ts).normalize()

    if len(work):
        template_df = work.tail(1).copy().reset_index(drop=True)
    else:
        template_df = pd.DataFrame([{c: np.nan for c in team_df.columns}])

    if "season" in template_df.columns:
        template_df.at[0, "season"] = int(season)
    if "game_date" in template_df.columns:
        template_df.at[0, "game_date"] = game_date_norm
    if "dh_index" in template_df.columns:
        template_df.at[0, "dh_index"] = int(dh_index)
    if "game_id" in template_df.columns:
        template_df.at[0, "game_id"] = _make_synthetic_game_id(
            work,
            season=int(season),
            game_date=game_date_norm,
            dh_index=int(dh_index),
            school_id=school_id,
        )
    if "result" in template_df.columns:
        template_df.at[0, "result"] = np.nan
    if school_id is not None and "school_id" in template_df.columns:
        template_df.at[0, "school_id"] = int(school_id)

    work = pd.concat([work, template_df], ignore_index=True, sort=False)

    work = prepare_team_df(work)
    work = add_pregame_averages(work)
    work = prev_stats_func(work)
    work = add_derived_features_next_snapshot(work)
    work = add_rolling_features(work)

    snap = work.loc[
        work["season"].eq(int(season))
        & pd.to_datetime(work["game_date"], errors="coerce").dt.normalize().eq(game_date_norm)
        & work["dh_index"].eq(int(dh_index))
    ].copy()

    if len(snap) == 0:
        return snap

    return snap.sort_values(["season", "game_date", "dh_index"]).tail(1).copy()


future_hosted_mask = (
    matchups["season"].eq(2026)
    & matchups["unc_score"].isna()
    & matchups["opponent_score"].isna()
)

# Fill preseason features for all remaining 2026 hosted games.
unc_pre_lookup = build_unc_preseason_lookup(unc_full, unc_prev_season)
matchups = matchups.merge(
    unc_pre_lookup,
    on="season",
    how="left",
    suffixes=("", "__unc_pre_fill"),
)

for c in unc_prev_season:
    fill_col = f"{c}__unc_pre_fill"
    if c in matchups.columns and fill_col in matchups.columns:
        matchups.loc[future_hosted_mask, c] = matchups.loc[future_hosted_mask, c].combine_first(
            matchups.loc[future_hosted_mask, fill_col]
        )

matchups = matchups.drop(
    columns=[f"{c}__unc_pre_fill" for c in unc_prev_season if f"{c}__unc_pre_fill" in matchups.columns],
    errors="ignore",
)

opp_pre_lookup = build_opp_preseason_lookup(
    opp,
    [c.replace("opp_", "", 1) for c in opp_prev_season if c.startswith("opp_")]
).rename(
    columns={
        c.replace("opp_", "", 1): c
        for c in opp_prev_season
        if c.startswith("opp_")
    }
)

matchups = matchups.merge(
    opp_pre_lookup,
    on=["opp_school_id", "season"],
    how="left",
    suffixes=("", "__opp_pre_fill"),
)

for c in opp_prev_season:
    fill_col = f"{c}__opp_pre_fill"
    if c in matchups.columns and fill_col in matchups.columns:
        matchups.loc[future_hosted_mask, c] = matchups.loc[future_hosted_mask, c].combine_first(
            matchups.loc[future_hosted_mask, fill_col]
        )

matchups = matchups.drop(
    columns=[f"{c}__opp_pre_fill" for c in opp_prev_season if f"{c}__opp_pre_fill" in matchups.columns],
    errors="ignore",
)

# Fill current-season midseason features for the next unplayed hosted game only.
future_rows = matchups.loc[future_hosted_mask].copy()
future_rows = future_rows.loc[future_rows["game_date"].notna()].copy()

if len(future_rows):
    next_idx = future_rows.sort_values(["game_date", "dh_index"]).index[0]
    next_row = matchups.loc[next_idx].copy()

    unc_next = build_next_team_snapshot(
        unc_full,
        season=int(next_row["season"]),
        game_date=next_row["game_date"],
        dh_index=int(next_row["dh_index"]),
        prev_stats_func=compute_previous_season_stats,
    )

    if len(unc_next):
        for c in unc_pregame_core + unc_derived + unc_recent:
            if c in matchups.columns and c in unc_next.columns:
                matchups.at[next_idx, c] = unc_next.iloc[0][c]

    if pd.notna(next_row["opp_school_id"]):
        opp_next = build_next_team_snapshot(
            opp,
            season=int(next_row["season"]),
            game_date=next_row["game_date"],
            dh_index=int(next_row["dh_index"]),
            prev_stats_func=compute_previous_season_stats_by_team,
            school_id=int(next_row["opp_school_id"]),
        )

        if len(opp_next):
            for c in opp_pregame_core + opp_derived + opp_recent:
                raw_c = c.replace("opp_", "", 1)
                if c in matchups.columns and raw_c in opp_next.columns:
                    matchups.at[next_idx, c] = opp_next.iloc[0][raw_c]

if {"team_strength", "opp_team_strength"}.issubset(matchups.columns):
    matchups["strength_diff"] = matchups["team_strength"] - matchups["opp_team_strength"]

if {"win_pct", "opp_win_pct"}.issubset(matchups.columns):
    matchups["win_pct_diff"] = matchups["win_pct"] - matchups["opp_win_pct"]

if "has_opponent_features" not in matchups.columns:
    matchups["has_opponent_features"] = (~matchups["opp_team_strength"].isna()).astype(int)
else:
    matchups["has_opponent_features"] = (~matchups["opp_team_strength"].isna()).astype(int)


final_feature_table = matchups[merge_keys + ordered_feature_cols].copy()

final_feature_table.info()



In [ ]:
# Join the final feature set back onto the cleaned GoHeels hosted dataset
# to mirror final_game_stat_scraper.
goheels_hosted_no_nonunc = goheels_hosted_clean.copy()

goheels_hosted_no_nonunc = goheels_hosted_no_nonunc[
    goheels_hosted_no_nonunc["season"].between(2013, 2026)
].copy()

for df_ in (goheels_hosted_no_nonunc, final_feature_table):
    df_["season"] = df_["season"].astype(int)
    df_["dh_index"] = df_["dh_index"].astype(int)
    df_["game_date"] = pd.to_datetime(df_["game_date"], errors="coerce").dt.normalize()

goheels_with_features = goheels_hosted_no_nonunc.merge(
    final_feature_table,
    on=merge_keys,
    how="left",
    validate="one_to_one",
)

completed = goheels_with_features["unc_score"].notna() & goheels_with_features["opponent_score"].notna()

missing_game_id_completed = goheels_with_features.loc[
    completed & (goheels_with_features["game_id"].isna() | (goheels_with_features["game_id"].astype("Int64") <= 0))
]

print("Rows:", goheels_with_features.shape)
print("Completed games missing/invalid game_id after join:", len(missing_game_id_completed))

id_cols = ["season", "game_date", "dh_index", "opponent"]
other_cols = [c for c in goheels_with_features.columns if c not in id_cols]
goheels_with_features = goheels_with_features[id_cols + other_cols]

goheels_with_features.info()


In [ ]:
FINAL_CSV.parent.mkdir(parents=True, exist_ok=True)
goheels_with_features.to_csv(FINAL_CSV, index=False)
print(f"saved final joined export -> {FINAL_CSV}")
